# PDR database viewer

Browse `PDR_database.fits`: the observed absorption systems used by PDRanalyser.

The file has four tables, each read as a pandas DataFrame:

| table | one row per | main columns |
|---|---|---|
| `db.systems` | system | `sys_id`, `name`, `z_em`, `z_dla`, `telescope`, `year`, `ref`, ... |
| `db.components` | velocity component | `sys_id`, `comp`, `z`, `z_err` |
| `db.values` | quantity (N, T01, n, Me, ...) | `sys_id`, `comp` (-1 = whole system), `name`, `val`, `plus`, `minus`, `type`, `format` |
| `db.groups` | system in a group | `group`, `sys_id`, `comps` (`all` or a list like `0,2`) |

`type`: `m` measurement, `u` upper limit, `l` lower limit, `f` fixed (no error). `format`: `log` or `dec`.

`PDR_database_public.fits` contains the published systems (see README).

In [ ]:
import copy
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

# folder with PDR_database.py and PDR_database.fits (change if the notebook is elsewhere)
PDR_FOLDER = os.path.abspath('.')
sys.path.insert(0, PDR_FOLDER)
from PDR_database import PDRdatabase

FILE = os.path.join(PDR_FOLDER, 'PDR_database_public.fits')   # or your own database file
db = PDRdatabase.read(FILE)
db

## 1. Groups

Number of systems and components in each group, and their redshift range.

In [ ]:
def group_summary(db):
    ncomp_all = db.components.groupby('sys_id').size()
    z = db.systems.set_index('sys_id')['z_dla']
    rows = []
    for g in db.groups_list():
        m = db.groups[db.groups['group'] == g]
        ncomp = sum(int(ncomp_all.get(r.sys_id, 0)) if r.comps == 'all' else len([x for x in r.comps.split(',') if x])
                    for r in m.itertuples())
        rows.append(dict(group=g, systems=len(m), components=ncomp,
                         z_dla_min=z[m['sys_id']].min(), z_dla_max=z[m['sys_id']].max()))
    return pd.DataFrame(rows).set_index('group')

group_summary(db)

## 2. The four tables

In [ ]:
display(db.systems.head())
display(db.components.head())
display(db.values.head())
display(db.groups.head())

## 3. Which quantities are in the database

How many systems / components have each quantity (sorted by the number of systems).

In [ ]:
level = np.where(db.values['comp'] < 0, 'systems', 'components')
quantities = (db.values.assign(level=level)
              .groupby(['name', 'level']).size().unstack(fill_value=0)
              .reindex(columns=['systems', 'components'], fill_value=0)
              .sort_values(['systems', 'components'], ascending=False))
quantities.head(40)

## 4. One group as a table

`db.table(group, names)` gives one row per system; `comp=True` gives one row per component;
`errors=True` adds `<name>_plus`, `<name>_minus` and `<name>_type` columns.

Group names are the full names from `db.groups_list()` (e.g. `QSO_Klimenko2024`); `pick_group('QSO')` returns the first group whose name starts with the prefix.

In [ ]:
def pick_group(db, prefix):
    """first group whose name starts with prefix, e.g. pick_group(db, 'QSO') -> 'QSO_Klimenko2024'"""
    names = [g for g in db.groups_list() if g.startswith(prefix)]
    if not names:
        raise KeyError(f'no group starting with "{prefix}"; groups: {db.groups_list()}')
    return names[0]

GROUP = pick_group(db, 'QSO')
print(GROUP)
db.table(GROUP, ['HI', 'H2', 'f', 'Me', 'CI', 'CO', 'T01'])

In [ ]:
db.table(GROUP, ['H2', 'H2j0', 'H2j1', 'H2j2', 'CI', 'T01'], comp=True, errors=True)

## 5. One system in detail

All values of a system: the whole-system values and each component, written with their errors and limits.

In [ ]:
def fmt(r):
    """value as text: 20.44 ± 0.03, 20.29 +0.02 -0.03, < 13.1, > 14.5"""
    if r['type'] == 'u':
        return f"< {r['val']:g}"
    if r['type'] == 'l':
        return f"> {r['val']:g}"
    if r['type'] == 'f' or (r['plus'] == 0 and r['minus'] == 0):
        return f"{r['val']:g}"
    if r['plus'] == r['minus']:
        return f"{r['val']:g} ± {r['plus']:g}"
    return f"{r['val']:g} +{r['plus']:g} -{r['minus']:g}"


def show_system(db, name):
    ids = db.find(name)
    if not ids:
        print(f'no system with "{name}" in its name')
        return
    if len(ids) > 1:
        print(f'several systems match "{name}": {ids}; showing {ids[0]}')
    sid = ids[0]
    info = db.systems.set_index('sys_id').loc[sid]
    print(f"{sid}   z_em = {info['z_em']}   z_dla = {info['z_dla']}")
    print('groups:', ', '.join(db.groups.loc[db.groups['sys_id'] == sid, 'group']))
    display(info[info.astype(str) != ''].to_frame('value'))
    display(db.components[db.components['sys_id'] == sid].set_index('comp'))
    v = db.values[(db.values['sys_id'] == sid) & (db.values['name'] != 'size')].copy()
    v['text'] = v.apply(fmt, axis=1)
    v['where'] = np.where(v['comp'] < 0, 'system', 'comp ' + v['comp'].astype(str))
    order = list(dict.fromkeys(v['name']))
    cols = list(dict.fromkeys(v.sort_values('comp')['where']))
    display(v.pivot_table(index='name', columns='where', values='text', aggfunc='first')
             .reindex(index=order, columns=cols).fillna(''))


show_system(db, 'J0000+0048')

## 6. Selecting systems with pandas

Example: systems with log N(H2) > 19 and metallicity > -1, measurements only (no limits).

In [ ]:
t = db.table(None, ['HI', 'H2', 'Me'], errors=True)
selected = t[(t['H2'] > 19) & (t['Me'] > -1) & (t['H2_type'] == 'm') & (t['Me_type'] == 'm')]
selected[['name', 'z_dla', 'HI', 'H2', 'Me']]

## 7. Plot: N(H2) versus N(HI)

Measurements with error bars; upper limits on N(H2) as downward triangles.

In [ ]:
# all groups in the file; replace with e.g. ['QSO_Klimenko2024', 'MW_CO_Klimenko2024'] to choose
GROUPS = db.groups_list()

fig, ax = plt.subplots(figsize=(7, 5))
for g in GROUPS:
    t = db.table(g, ['HI', 'H2'], errors=True).dropna(subset=['HI', 'H2'])
    t = t[(t['HI'] > 0) & (t['H2'] > 0)]
    meas = t[t['H2_type'].isin(['m', 'f']) & t['HI_type'].isin(['m', 'f'])]
    upper = t[t['H2_type'] == 'u']
    eb = ax.errorbar(meas['HI'], meas['H2'],
                     xerr=[meas['HI_minus'], meas['HI_plus']], yerr=[meas['H2_minus'], meas['H2_plus']],
                     fmt='o', ms=4, lw=0.8, label=f'{g} ({len(meas)})')
    if len(upper):
        ax.scatter(upper['HI'], upper['H2'], marker='v', s=25, color=eb[0].get_color())
ax.set_xlabel('log N(HI)')
ax.set_ylabel('log N(H$_2$)')
ax.legend()
plt.show()

## 8. Editing

The edits below are made on a copy (`ed`), so `db` is unchanged. To keep changes, edit `db` and save it with
`db.write(...)`. Values are stored once per system, so an edit changes the system in every group that contains it.

In [ ]:
ed = copy.deepcopy(db)

# change a value of the whole system, and of a component
ed.set_value('J0000+0048', 'H2', 20.45, 0.03)
ed.set_value('J0000+0048', 'H2j0', 20.30, 0.02, comp=4)

# add a new system with one component
ed.add_system('J1234+5678', z_em=2.9, z_dla=2.41, telescope='VLT/ESPRESSO', ref=['Klimenko2026'])
c = ed.add_component('J1234+5678', z=2.4105)
ed.set_value('J1234+5678', 'H2', 19.2, 0.1, comp=c)
ed.set_value('J1234+5678', 'CO', 13.0, comp=c, type='u')     # upper limit
ed.set_value('J1234+5678', 'T01', 85, 5, comp=c, format='dec')

# make a new group
ed.add_group('my_sample', ['J0000+0048', 'J1234+5678'])
ed.add_to_group('my_sample', 'J1237+0647', comps=[0])        # only component 0

display(group_summary(ed).tail(3))
show_system(ed, 'J1234+5678')

# ed.write(os.path.join(PDR_FOLDER, 'PDR_database_edited.fits'))   # uncomment to save

## 9. Export a table

For example, to open a group in Excel or share it as CSV (uncomment to write the files).

In [ ]:
out = db.table(GROUP, ['HI', 'H2', 'Me'], errors=True)
# out.to_csv(f'{GROUP}.csv')
# out.to_excel(f'{GROUP}.xlsx')
out.head()